# DocTrack — Générer la voix wolof de l'assistant (Adia_TTS, CONCREE)

**Adia_TTS** est un réseau de neurones entraîné sur **40 heures de voix wolof** : il *génère* un signal audio nouveau à partir d'un texte
(ce ne sont pas des enregistrements collés bout à bout). Licence **Apache 2.0** : utilisable aussi en exploitation payante
(l'ancien modèle xTTS-v2-wolof était limité à un usage non commercial).

L'assistant utilise toujours les mêmes phrases (« Ban kayit moo la réer ? »…). On les fait lire **une seule fois**, puis on dépose les MP3
dans DocTrack : la voix est ensuite instantanée et gratuite.

**Deux façons de faire, même résultat :**
- **ce notebook** (carte graphique gratuite de Colab) : 2 à 5 secondes par phrase, tout en 3 minutes — pratique pour essayer plusieurs voix ;
- **le service `doctrack-tts`** sur Hugging Face + `php artisan doctrack:assistant-voices` : rien à télécharger, mais 30 s à 2 min par phrase.

**Avant de commencer**
1. **Exécution → Modifier le type d'exécution → GPU T4**.
2. Sur votre PC, dans le dossier de l'API : `php artisan doctrack:assistant-phrases --lang=wo`
   → fichier `storage/app/assistant/phrases_wo.json` à envoyer à l'étape 3.

## 1. Installation (3 à 5 minutes)

In [ ]:
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile pydub
!apt-get -qq install -y ffmpeg > /dev/null
import torch
print('GPU :', torch.cuda.is_available())

## 2. Charger le modèle (≈ 3,5 Go)

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
MODEL_ID = 'CONCREE/Adia_TTS'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:    # la description de la voix a son propre tokenizer (celui de l'encodeur de texte)
    description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
except Exception:
    description_tokenizer = tokenizer
RATE = model.config.sampling_rate
print('Modèle chargé ✔', RATE, 'Hz')

## 3. Envoyer `phrases_wo.json`

In [ ]:
from google.colab import files
import json
up = files.upload()
phrases = json.load(open(next(n for n in up if n.endswith('.json')), encoding='utf-8'))
print(len(phrases), 'phrases')

## 4. Préparation du texte et réglage de la voix

Même préparation que le service `doctrack-tts` : « DocTrack » prononcé « Dok Trak », nombres en lettres, point final ajouté si besoin.

**Contre les phrases coupées avant la fin :**
- le texte est découpé en segments courts (110 caractères) : le modèle s'arrête trop tôt sur les longs ;
- chaque segment peut durer jusqu'à ~29 secondes (avant : ~11 s) ;
- si l'audio est trop court pour la longueur du texte, il est refait avec une autre graine, et on garde le plus complet.

**Réglages** : `VOIX` (description en anglais) et `GRAINE` (même graine = même voix). L'étape 5 permet de les comparer à l'oreille.

In [ ]:
import re
import numpy as np
SEGMENT_CHARS = 110          # segments courts : le modèle va jusqu'au bout de la phrase
MAX_NEW_TOKENS = 2500        # ≈ 29 s d'audio par segment (1000 coupait vers 11 s)
MIN_SECONDS_PER_CHAR = 0.045 # en dessous de cette durée, la phrase a été coupée
ESSAIS = 4

_UNITS = ["tus", "benn", "ñaar", "ñett", "ñeent", "juróom", "juróom-benn", "juróom-ñaar", "juróom-ñett", "juróom-ñeent"]
# Prononciations à forcer (le modèle lit le wolof, pas les sigles ni les mots étrangers).
_SPOKEN = {
    "doctrack": "Dok Trak",
    "sms": "ès-èm-ès",
    "qr": "ki-èr",
    "cni": "sé-èn-i",
}


def wolof_number(n: int) -> str:
    """Nombre en toutes lettres (0 à 999 999), système décimal wolof : fukk, téeméer, junni."""
    if n < 10:
        return _UNITS[n]
    if n < 20:
        return "fukk" if n == 10 else f"fukk ak {_UNITS[n - 10]}"
    if n < 100:
        tens, rest = divmod(n, 10)
        head = f"{_UNITS[tens]}-fukk"
        return head if rest == 0 else f"{head} ak {_UNITS[rest]}"
    if n < 1000:
        hundreds, rest = divmod(n, 100)
        head = "téeméer" if hundreds == 1 else f"{_UNITS[hundreds]} téeméer"
        return head if rest == 0 else f"{head} ak {wolof_number(rest)}"
    thousands, rest = divmod(n, 1000)
    head = "junni" if thousands == 1 else f"{wolof_number(thousands)} junni"
    return head if rest == 0 else f"{head} ak {wolof_number(rest)}"


def _spell_digits(match: re.Match) -> str:
    digits = match.group(0)
    # Numéros longs (téléphone, référence) : chiffre par chiffre ; sinon le nombre en toutes lettres.
    if len(digits) > 6 or (len(digits) > 1 and digits.startswith("0")):
        return ", ".join(_UNITS[int(d)] for d in digits)
    return wolof_number(int(digits))


def normalize(text: str) -> str:
    """Texte prêt à être lu : sigles prononcés, chiffres en lettres, espaces et ponctuation propres."""
    text = text.replace("’", "'").replace(" ", " ").replace(" ", " ")
    text = re.sub(r"[«»\"“”]", "", text)
    text = re.sub(r"\b\w+\b", lambda m: _SPOKEN.get(m.group(0).lower(), m.group(0)), text)
    text = re.sub(r"(?<=\d)[ .](?=\d{3}\b)", "", text)          # 3 000 → 3000
    text = re.sub(r"\d+", _spell_digits, text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)                 # « Lan nga bëgg def ? » → « def? »
    text = re.sub(r"\s+", " ", text).strip()
    # Sans ponctuation finale, le modèle ne « pose » pas la voix et avale la dernière syllabe.
    return text if not text or text[-1] in ".!?…" else text + "."


def split_segments(text: str, limit: int = SEGMENT_CHARS) -> list[str]:
    """Découpe aux frontières naturelles (phrases, puis virgules, puis mots) sans dépasser la limite du modèle."""
    sentences = [s.strip() for s in re.split(r"(?<=[.!?…])\s+", text) if s.strip()]
    pieces: list[str] = []
    for sentence in sentences:
        if len(sentence) <= limit:
            pieces.append(sentence)
            continue
        current = ""
        for part in re.split(r"(?<=[,;:])\s+", sentence):
            while len(part) > limit:                             # très long sans virgule : coupe au dernier mot
                cut = part.rfind(" ", 0, limit)
                cut = cut if cut > 0 else limit
                if current:
                    pieces.append(current)
                    current = ""
                pieces.append(part[:cut].strip())
                part = part[cut:].strip()
            if current and len(current) + 1 + len(part) > limit:
                pieces.append(current)
                current = part
            else:
                current = f"{current} {part}".strip()
        if current:
            pieces.append(current)
    # Regroupe les phrases courtes voisines : moins d'appels au modèle, enchaînement plus naturel.
    merged: list[str] = []
    for piece in pieces:
        if merged and len(merged[-1]) + 1 + len(piece) <= limit:
            merged[-1] = f"{merged[-1]} {piece}"
        else:
            merged.append(piece)
    return merged


VOIX = 'A clear and educational voice, with a flow adapted to learning'
GRAINE = 42

def _une_fois(segment, voix, graine):
    torch.manual_seed(graine)
    d = description_tokenizer(voix, return_tensors='pt').to(device)
    p = tokenizer(segment, return_tensors='pt').to(device)
    with torch.inference_mode():
        audio = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                               prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask,
                               do_sample=True, temperature=0.8, top_k=50, repetition_penalty=1.2,
                               max_new_tokens=MAX_NEW_TOKENS)
    onde = audio.cpu().numpy().squeeze().astype(np.float32)
    crete = float(np.max(np.abs(onde))) if onde.size else 0.0
    return onde / crete * 0.95 if crete > 0 else onde

def complet(segment, secondes):
    return secondes >= max(0.6, len(segment) * MIN_SECONDS_PER_CHAR)

def lire(texte, voix=None, graine=None, detail=False):
    """Lit un texte. Un segment coupé avant la fin est refait avec une autre graine ; on garde le plus complet."""
    voix, graine = voix or VOIX, GRAINE if graine is None else graine
    morceaux, douteux = [], []
    for i, segment in enumerate(split_segments(normalize(texte))):
        meilleur = None
        for essai in range(ESSAIS):
            onde = _une_fois(segment, voix, graine + essai)
            if meilleur is None or len(onde) > len(meilleur): meilleur = onde
            if complet(segment, len(onde) / RATE): break
        if not complet(segment, len(meilleur) / RATE): douteux.append(segment)
        if i: morceaux.append(np.zeros(int(RATE * 0.25), dtype=np.float32))
        morceaux.append(meilleur)
    morceaux.append(np.zeros(int(RATE * 0.2), dtype=np.float32))   # petit silence final : la dernière syllabe n'est pas rognée
    onde = np.concatenate(morceaux)
    return (onde, douteux) if detail else onde

## 5. Choisir la voix et le ton (écoutez avant de tout générer)

Le **ton** dépend surtout de trois choses : la description de la voix, la graine, et **le texte lui-même**
(un wolof écrit comme on le parle, avec la ponctuation aux endroits où l'on respire, est lu avec une mélodie plus naturelle).

La cellule lit la même phrase avec plusieurs voix. Notez celle qui sonne le plus « wolof » et reportez-la dans `VOIX` et `GRAINE` juste en dessous.

In [ ]:
from IPython.display import Audio, display
ESSAI = next((p['text'] for p in phrases if p['key'] == 'welcome'), phrases[0]['text'])
VOIX_A_COMPARER = [
    'A clear and educational voice, with a flow adapted to learning',
    'A warm and natural voice, speaking at a moderate pace, close and clear recording',
    'A calm and friendly voice with an expressive, lively tone, speaking slowly and clearly',
]
print(ESSAI)
for v in VOIX_A_COMPARER:
    for g in (42, 7):
        print(f"\nVOIX = {v!r}\nGRAINE = {g}")
        display(Audio(lire(ESSAI, voix=v, graine=g), rate=RATE))

In [ ]:
# Votre choix (copiez la description et la graine préférées)
VOIX = 'A clear and educational voice, with a flow adapted to learning'
GRAINE = 42

## 6. Générer toutes les phrases → MP3

À la fin, la liste **« à réécouter »** donne les phrases dont un segment est resté trop court malgré les nouveaux essais.

In [ ]:
import os, soundfile as sf
from pydub import AudioSegment, silence
os.makedirs('audio_wo', exist_ok=True)

def nettoie(wav_path, mp3_path):
    son = AudioSegment.from_wav(wav_path)
    debut = silence.detect_leading_silence(son, silence_threshold=-45)
    # Fin : seuil très bas (-60 dB) pour ne pas rogner une dernière syllabe douce, et 250 ms gardés.
    fin = silence.detect_leading_silence(son.reverse(), silence_threshold=-60)
    son = son[max(0, debut - 80): len(son) - max(0, fin - 250)]
    son.set_channels(1).export(mp3_path, format='mp3', bitrate='64k')
    return len(son) / 1000

def genere(p, voix=None, graine=None):
    wav = f"audio_wo/{p['key']}.wav"
    onde, douteux = lire(p['text'], voix=voix, graine=graine, detail=True)
    sf.write(wav, onde, RATE)
    duree = nettoie(wav, f"audio_wo/{p['key']}.mp3"); os.remove(wav)
    return duree, douteux

a_reecouter = []
for i, p in enumerate(phrases, 1):
    duree, douteux = genere(p)
    if douteux: a_reecouter.append(p['key'])
    print(f"{i}/{len(phrases)}  {p['key']:<32} {duree:5.1f} s  {'⚠ à réécouter' if douteux else ''}")
print('\nÀ réécouter :', a_reecouter or 'aucune')

## 7. Tout écouter (texte sous les yeux)

Écoutez chaque phrase **jusqu'au dernier mot**. Notez les clés de celles qui sont coupées ou mal dites.

In [ ]:
for p in phrases:
    print(p['key'], '—', p['text']); display(Audio(f"audio_wo/{p['key']}.mp3"))

## 8. Refaire seulement certaines phrases

Mettez les clés dans `A_REFAIRE` et changez la graine (chaque graine donne une autre lecture). Relancez autant de fois que nécessaire.

In [ ]:
A_REFAIRE = a_reecouter          # ou par exemple ['welcome', 'ask_doc_type_lost']
NOUVELLE_GRAINE = 123
for p in phrases:
    if p['key'] in A_REFAIRE:
        duree, douteux = genere(p, graine=NOUVELLE_GRAINE)
        print(p['key'], f"{duree:.1f} s", '⚠' if douteux else '✔', '—', p['text'])
        display(Audio(f"audio_wo/{p['key']}.mp3"))

## 9. Télécharger le ZIP

In [ ]:
!rm -f audio_wo.zip && cd audio_wo && zip -q -r ../audio_wo.zip *.mp3
files.download('audio_wo.zip')

## 10. Installer les fichiers dans DocTrack

1. Décompressez `audio_wo.zip` dans le dossier de l'API : **`storage/app/assistant/audio/wo/`** en **remplaçant** les anciens fichiers
   (un fichier par phrase : `welcome.mp3`, `ask_doc_type_lost.mp3`…).
2. `php artisan doctrack:assistant-voices --adopt` : DocTrack note, pour chaque nouveau fichier, le texte qu'il prononce
   (inutile si vous remplacez seulement des fichiers déjà enregistrés et que le texte n'a pas changé).
3. Vérification : écran **Administration → Voix de l'assistant** (écoute phrase par phrase) ou `php artisan doctrack:assistant-voices --status`.

**Le ton ne convient toujours pas ?** Faites relire les textes wolof de `config/assistant.php` par un locuteur natif
(phrases courtes, tournures parlées, pas de sigle ni de mot français), puis régénérez. Pour une voix 100 % naturelle,
une personne peut aussi enregistrer elle-même les phrases : déposez ses fichiers nommés `{clé}.mp3` dans le même dossier, puis `--adopt`.